In [1]:
"""Run this to establish the connection with the database"""
# load the SQL extension
%load_ext sql

# create the connection string
import os
user = os.getenv('postgres_user')
password = os.getenv('postgres_password')
db = os.getenv('postgres_db')
db_url = f"postgresql://{user}:{password}@db/{db}"

# establish the connection
from sqlalchemy import create_engine
engine = create_engine(db_url)

# set %sql magic commands to use the connection
%sql engine

In [87]:
%%sql
select
    name,
    pct_snap_retailers
from county_analytics_view
    where ct_snap_retailers > 20
order by 2 desc

Running query in 'postgresql://healthgps:***@db/healthgps'

1842 rows affected.

name,pct_snap_retailers
TIFT,100.0000000000000000
GREEN,100.0000000000000000
KUSILVAK,100.0000000000000000
DE SOTO,100.0000000000000000
HOCKLEY,100.0000000000000000
MADISON,100.0000000000000000
WASHINGTON,100.0000000000000000
JUNEAU,100.0000000000000000
BROOKE,100.0000000000000000
PRENTISS,100.0000000000000000


In [88]:
"""You can optionally pull this into a pandas dataframe"""
results_df = %sql select * from food_posts

Running query in 'postgresql://healthgps:***@db/healthgps'

51732 rows affected.

In [6]:
%%sql
-- Check for duplicates between SNAP retailers and access food pantries
SELECT
    a."Store_Name" as id,
    b.name as pantry_name,
    st_distance(a.geometry, b.geometry::geography, False) as dist
FROM snap_retailers_raw a
CROSS JOIN LATERAL (
  SELECT
    "locationName" as name,
    pantries.geometry
  FROM access_food_foodbanks_raw pantries
  ORDER BY
         pantries.geometry <-> a.geometry -- nearest neighbor
  LIMIT  1
) AS b
order by 3 asc
limit 1000

Running query in 'postgresql://healthgps:***@db/healthgps'

1000 rows affected.

id,pantry_name,dist
Metro Atlanta Urban Farm,Metro Atlanta Urban Farm in College Park Inc.,0.38149356
Gipson Grocery,Southside Economic & Community Dev Corp,1.12890312
DOLLARTREE 10245,Nations Church,1.38239265
Hiouchi Market,Hiouchi Hamlet,1.99638061
Southeastern Farmers Market,Southeastern Church of Christ,2.0538576
For Oak Cliff Farmers Market NAFMNP,For Oak Cliff,2.06219117
Harar Market,Colorado Muslim Community Center,2.24415485
Brighton Winter Farmer's Market NAFMNP,ABCD/Allston Brighton,2.3780434
Gric Market 817,Gila Indian River Community,2.84057335
Muncie Common Market,The Common Market,3.36015309


In [28]:
sql = """select
    a.id as a_id,
    b.id as b_id,
    ARRAY(SELECT element FROM unnest(ARRAY[a.id, b.id]) AS element ORDER BY element),
    a.name as a_name,
    b.name as b_name,
    a.address as a_address,
    b.address as b_address,
    a.phone_number_1 as a_pn_1,
    b.phone_number_1 as b_pn_1,
    a.phone_number_2 as a_pn_2,
    b.phone_number_2 as b_pn_2,
    a.source as a_source,
    b.source as b_source,
    st_distance(a.geometry::geography, b.geometry::geography) as dist
from food_posts a
cross join lateral (
    select
        id,
        name,
        address,
        phone_number_1,
        phone_number_2,
        source,
        geometry
    from food_posts f
    where f.id <> a.id
    order by f.geometry <-> a.geometry
    limit 1
) b"""

In [29]:
df = pandas.read_sql(sql,con=engine)

In [37]:
df[(df['dist']<10) & (df['a_name']==df['b_name']) | (df['a_pn_1']==df['b_pn_1'])]

,a_id,b_id,array,a_name,b_name,a_address,b_address,a_pn_1,b_pn_1,a_pn_2,b_pn_2,a_source,b_source,dist
0,c9277530-a3b1-4fd9-8c7c-8244f1fcde42,1e2133f7-8565-4404-b2cc-53557389e970,"[1e2133f7-8565-4404-b2cc-53557389e970, c927753...",BREATH OF GOD FEEDING PROGRAM,BREATH OF GOD FEEDING PROGRAM,None,"141 SOUTH CLINTON STREET, BALTIMORE, MARYLAND,...",None,410-675-5616,None,None,md_food_bank,why_hunger_api,1.286851
1,cfd1e907-00a7-494c-aeea-55381c5e347d,b6655aac-b17f-40c0-a729-305be05b2131,"[b6655aac-b17f-40c0-a729-305be05b2131, cfd1e90...",SOUTH CREEK COMMUNITY DEVELOPMENT CORPORATION,SOUTH CREEK COMMUNITY DEVELOPMENT CORPORATION,None,None,None,None,None,None,md_food_bank,md_food_bank,0.000000
2,b6655aac-b17f-40c0-a729-305be05b2131,cfd1e907-00a7-494c-aeea-55381c5e347d,"[b6655aac-b17f-40c0-a729-305be05b2131, cfd1e90...",SOUTH CREEK COMMUNITY DEVELOPMENT CORPORATION,SOUTH CREEK COMMUNITY DEVELOPMENT CORPORATION,None,None,None,None,None,None,md_food_bank,md_food_bank,0.000000
5,24bc3a1b-3339-4191-992d-789334547e36,440ef1c3-e9cf-4b7e-9d48-d9dae8fc95e6,"[24bc3a1b-3339-4191-992d-789334547e36, 440ef1c...",LIFE CHURCH MINISTRIES,LIFE CHURCH MINISTRIES,None,"3820 SOUTH HANOVER STREET, BALTIMORE, MARYLAND...",None,443-708-8537,None,None,md_food_bank,why_hunger_api,1.531069
11,9aedcafa-5244-442e-a931-8f52f3fadd25,17f90922-0ab8-4b70-bca8-583fec87ea99,"[17f90922-0ab8-4b70-bca8-583fec87ea99, 9aedcaf...",MACEDONIA PROJECT,MACEDONIA PROJECT,None,"5401 FRANKFORD AVENUE, BALTIMORE, MARYLAND, 21...",None,410-488-5650,None,None,md_food_bank,why_hunger_api,0.389869
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51724,b39fe883-c7b5-40e1-ae41-c46195bef6ec,051b958e-1771-4729-ab19-e61cf8c50da1,"[051b958e-1771-4729-ab19-e61cf8c50da1, b39fe88...",BYLAS FOOD DISTRIBUTION,BYLAS FOOD DISTRIBUTION PROGRAM,"RAINBOW CITY CIR #2, SAN CARLOS, AZ, 85550","RAINBOW CITY CIRCLE BUILDING 2, BYLAS, AZ, 85...",928-475-2304,928-475-2304,None,None,access_food_api,access_food_api,36.806348
51725,c7a4064c-c364-4a5a-ad08-ad238b102f71,c401ed4c-7774-43a4-80c5-62dc782911ba,"[c401ed4c-7774-43a4-80c5-62dc782911ba, c7a4064...",FLAGSTAFF FAMILY FOOD CENTER: SUMMIT HIGH SCHOOL,FLAGSTAFF FAMILY FOOD CENTER: WAREHOUSE FOOD BANK,"4000 N CUMMINGS ST, FLAGSTAFF, AZ, 86004","3805 E HUNTINGTON DR, FLAGSTAFF, AZ, 86004",928-526-2211,928-526-2211,928-526-2211,928-526-2211,access_food_api,access_food_api,1575.591811
51728,dedf8a09-4fc1-4277-9fb0-d51f161d9b16,b5f7de8a-4664-402a-a855-e45c4cf43de8,"[b5f7de8a-4664-402a-a855-e45c4cf43de8, dedf8a0...",SMFB AT KAIBAB PAIUTE,SMFB AT KAIBAB (TURKEY DISTRIBUTION),"600 N PINE SPRINGS RD, FREDONIA, AZ, 86022","600 N PINE SPRINGS RD, FREDONIA, AZ, 86022",928-220-5335,928-220-5335,928-220-5335,928-220-5335,access_food_api,access_food_api,0.242619
51729,b5f7de8a-4664-402a-a855-e45c4cf43de8,dedf8a09-4fc1-4277-9fb0-d51f161d9b16,"[b5f7de8a-4664-402a-a855-e45c4cf43de8, dedf8a0...",SMFB AT KAIBAB (TURKEY DISTRIBUTION),SMFB AT KAIBAB PAIUTE,"600 N PINE SPRINGS RD, FREDONIA, AZ, 86022","600 N PINE SPRINGS RD, FREDONIA, AZ, 86022",928-220-5335,928-220-5335,928-220-5335,928-220-5335,access_food_api,access_food_api,0.242619


In [50]:
df23 = pandas.read_csv('cps_fss_dec23.csv')
df22 = pandas.read_csv('cps_fss_dec22.csv')
df21 = pandas.read_csv('cps_fss_dec21.csv')
df20 = pandas.read_csv('cps_fss_dec20.csv')
df19 = pandas.read_csv('cps_fss_dec19.csv')

In [52]:
df = pandas.concat([df19, df20, df21, df22, df23])

In [53]:
df

,HRHHID,HRMONTH,HRYEAR4,HURESPLI,HUFINAL,HULANGCODE,HETENURE,HEHOUSUT,HETELHHD,HETELAVL,...,PEPAR1,PEPAR2TYP,PEPAR1TYP,PXPAR2,PXPAR1,PXPAR2TYP,PXPAR1TYP,HESP7_1,HESC4A,PRERNMIN
0,581125017600866,12,2019,2,201,0.0,1,1,1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,581125017600866,12,2019,2,201,0.0,1,1,1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,581125017600866,12,2019,2,201,0.0,1,1,1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,581125017600866,12,2019,2,201,0.0,1,1,1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,809001585510960,12,2019,1,201,0.0,2,1,1,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126827,790456071508119,12,2023,2,201,NaN,1,1,1,-1,...,-1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0
126828,962070105399566,12,2023,1,201,NaN,1,1,1,-1,...,-1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0
126829,962070105399566,12,2023,1,201,NaN,1,1,1,-1,...,-1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0
126830,962070105399566,12,2023,1,201,NaN,1,1,1,-1,...,2.0,1.0,2.0,0.0,0.0,0.0,0.0,-1.0,-1.0,-1.0


In [63]:
df[df['HESC3']>0].groupby(['GTCO'])['HESC3'].count()

GTCO
0.0      78135
1.0       3319
3.0       4004
5.0       1405
7.0        322
         ...  
550.0       19
700.0       74
710.0       92
760.0       39
810.0      128
Name: HESC3, Length: 100, dtype: int64

In [65]:
len(df[df['HESC3']>0])

158940